In [1]:
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler

import matplotlib.pyplot as plt
import plotly.express as px

np.random.seed(42)

# Week 11: Clustering - DBSCAN and HAC

In [2]:
socioeconomic = pd.read_csv(r"..\Processed Data\USDA Socioeconomic Indicators.csv")

# Filter out national and state totals so we isolate just counties
socioeconomic = socioeconomic[(socioeconomic['FIPS_Code'] != 0) & (socioeconomic['FIPS_Code'] % 1000 != 0)]
socioeconomic

,FIPS_Code,State,Area_Name,Year,Civilian_labor_force,Employed,Med_HH_Income_Percent_of_State_Total,Median_Household_Income,Metro,Rural_Urban_Continuum_Code,Unemployed,Unemployment_rate,Urban_Influence_Code
48,1001,AL,"Autauga County, AL",2000,21861.0,20971.0,117.500000,70148.000000,1.0,2.0,890.0,4.071177,2.0
49,1001,AL,"Autauga County, AL",2001,22081.0,21166.0,117.500000,70148.000000,1.0,2.0,915.0,4.143834,2.0
50,1001,AL,"Autauga County, AL",2002,22161.0,21096.0,117.500000,70148.000000,1.0,2.0,1065.0,4.805740,2.0
51,1001,AL,"Autauga County, AL",2003,22695.0,21557.0,117.500000,70148.000000,1.0,2.0,1138.0,5.014320,2.0
52,1001,AL,"Autauga County, AL",2004,23241.0,22146.0,117.500000,70148.000000,1.0,2.0,1095.0,4.711501,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
78432,72153,PR,"Yauco Municipio, PR",2018,9748.0,8315.0,53.026793,36042.798069,1.0,2.0,1433.0,14.700451,2.0
78433,72153,PR,"Yauco Municipio, PR",2019,9851.0,8409.0,53.026793,36042.798069,1.0,2.0,1442.0,14.638108,2.0
78434,72153,PR,"Yauco Municipio, PR",2021,10205.0,9009.0,53.026793,36042.798069,1.0,2.0,1196.0,11.719745,2.0
78435,72153,PR,"Yauco Municipio, PR",2022,10311.0,9247.0,53.026793,36014.531895,1.0,2.0,1064.0,10.319077,2.0


### DBSCAN

##### Focusing on capstone years (2018-2020)

In [3]:
county_2018 = socioeconomic[socioeconomic['Year'] == 2018].copy()
county_2019 = socioeconomic[socioeconomic['Year'] == 2019].copy()
county_2020 = socioeconomic[socioeconomic['Year'] == 2020].copy()

len(county_2018), len(county_2019), len(county_2020)

(3219, 3219, 3142)

In [4]:
# CREDIT: Code adapted - DBSCAN tutorial referenced below from Greg Hogg
# https://www.youtube.com/watch?v=VO_uzCU_nKw

def get_scores_and_labels(combinations, X):
    scores = []
    all_labels_list = []


    for i, (eps, num_samples) in enumerate(combinations):
        dbscan_model = DBSCAN(eps=eps, min_samples=num_samples).fit(X)
        labels = dbscan_model.labels_
        labels_set = set(labels)
        num_clusters = len(labels_set)

        # get rid of noise cluster in count
        if -1 in labels_set:
            num_clusters -= 1

        # label poor models (not enough clusters or too many clusters with high complexity)
        if (num_clusters < 2) or (num_clusters > 50):
            scores.append(-10)
            all_labels_list.append('bad')
            continue

        # ignore noise cluster for scoring
        mask = labels != -1
        X_clean = X[mask]
        labels_clean = labels[mask]

        # Ensure we still have at least 2 real clusters remaining after removing noise
        if len(set(labels_clean)) < 2:
            scores.append(-10)
            all_labels_list.append('bad')
            continue

        # Append scores and labels
        scores.append(silhouette_score(X_clean, labels_clean, random_state=42)) # taking a sample to reduce runtime on laptop
        all_labels_list.append(labels)

    best_index = np.argmax(scores)
    best_parameters = combinations[best_index]
    best_labels = all_labels_list[best_index]
    best_score = scores[best_index]

    return {'best_epsilon': best_parameters[0],
            'best_min_samples': best_parameters[1],
            'best_labels': best_labels,
            'best_score': best_score}

[DBSCAN TUTORIAL REFERENCED](https://www.youtube.com/watch?v=VO_uzCU_nKw)

##### 2018 Data

In [5]:
epsilons = np.linspace(0.15, 0.35, 12)
epsilons

array([0.15      , 0.16818182, 0.18636364, 0.20454545, 0.22272727,
       0.24090909, 0.25909091, 0.27727273, 0.29545455, 0.31363636,
       0.33181818, 0.35      ])

In [6]:
min_samples = np.arange(10, 30, 3)
min_samples

array([10, 13, 16, 19, 22, 25, 28])

In [7]:
import itertools

combinations = list(itertools.product(epsilons, min_samples))
N = len(combinations)
N

84

In [8]:
X_features = pd.DataFrame({
    'Unemploymnt_rate': county_2018['Unemployment_rate'],
    'log_Income': np.log1p(county_2018['Median_Household_Income'])
})


# Scale data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_features)

best_dict = get_scores_and_labels(combinations, X_scaled)

print("Best Epsilon:", best_dict['best_epsilon'])
print("Best Min Samples:", best_dict['best_min_samples'])
print("Best Silhouette Score:", best_dict['best_score'])

Best Epsilon: 0.20454545454545453
Best Min Samples: 13
Best Silhouette Score: 0.6409609715251665


In [9]:
best_dict

{'best_epsilon': np.float64(0.20454545454545453),
 'best_min_samples': np.int64(13),
 'best_labels': array([ 0,  0,  0, ..., -1, -1, -1], shape=(3219,)),
 'best_score': 0.6409609715251665}

In [10]:
dbscan_model = DBSCAN(eps=best_dict['best_epsilon'], min_samples=best_dict['best_min_samples']).fit(X_scaled)
dbscan_model

,eps,np.float64(0....4545454545453)
,min_samples,np.int64(13)
,metric,'euclidean'
,metric_params,None
,algorithm,'auto'
,leaf_size,30
,p,None
,n_jobs,None


In [11]:
county_2018['cluster'] = dbscan_model.labels_
county_2018['cluster'].value_counts()

cluster
 0    2957
-1     249
 1      13
Name: count, dtype: int64

In [12]:
unemployment_rate, median_household_income = county_2018['Unemployment_rate'], county_2018['Median_Household_Income']

In [13]:
fig = px.scatter(x=unemployment_rate, y=median_household_income, color=county_2018['cluster'].astype(str), opacity=0.4,
                 title="DBSCAN Clustering - U.S. County Unemployment Rate vs. Median Household Income (2018)",
                 labels= {'x': 'Unemployment Rate (%)', 'y': 'Median Household Income (USD)'})
# fig.update_traces(marker=dict(size=3))
fig.show()

In [14]:
# Check cluster counts and noise percentage
print("Cluster Counts:")
print(county_2018['cluster'].value_counts())

print("\nNoise Percentage:")
print(f"{(county_2018['cluster'] == -1).mean() * 100:.2f}%")

Cluster Counts:
cluster
 0    2957
-1     249
 1      13
Name: count, dtype: int64

Noise Percentage:
7.74%


In [15]:
# Group by cluster to see the socio-economic profile of each group
cluster_summary = county_2018.groupby('cluster').agg(
    County_Count=('FIPS_Code', 'count'),
    Avg_Unemployment_Rate=('Unemployment_rate', 'mean'),
    Avg_Median_Income=('Median_Household_Income', 'mean'),
    Min_Income=('Median_Household_Income', 'min'),
    Max_Income=('Median_Household_Income', 'max')
).reset_index()

cluster_summary

,cluster,County_Count,Avg_Unemployment_Rate,Avg_Median_Income,Min_Income,Max_Income
0,-1,249,8.038952,58593.504494,28972.000000,167605.0
1,0,2957,3.939337,63064.553768,36608.000000,124291.0
2,1,13,9.811146,35710.547329,35487.754305,36237.0


DBSCAN used on the USDA Socioeconomic Indicators dataset for 2018 clustered the data into 3 clusters, 0 being the vast majority of counties, 1 being outlier counties, and -1 being noise. Cluster 0 has an average unemployment rate of 3.94% and a median household income of $63,064. Cluster 1 represents severely economically distressed counties with elevated unemplyoment rates of 9.81% and average median household incomes of $35,710. Cluster -1, identifies outlier noise within the dataset which isolates extreme high income, extreme low umemployment, and extreme high unemployment. Cluster -1 has an average unemployment rate of 8.04% and an average median household income of $58,593. More hypertuning could be done in order to try to find better clusters so that there are more definable clusters.

In [16]:
epsilons = np.linspace(0.15, 0.3, 20)
epsilons

array([0.15      , 0.15789474, 0.16578947, 0.17368421, 0.18157895,
       0.18947368, 0.19736842, 0.20526316, 0.21315789, 0.22105263,
       0.22894737, 0.23684211, 0.24473684, 0.25263158, 0.26052632,
       0.26842105, 0.27631579, 0.28421053, 0.29210526, 0.3       ])

In [17]:
min_samples = np.arange(10, 30, 4)
min_samples

array([10, 14, 18, 22, 26])

In [18]:
combinations = list(itertools.product(epsilons, min_samples))
N = len(combinations)
N

100

In [19]:
X_features = pd.DataFrame({
    'Unemploymnt_rate': county_2018['Unemployment_rate'],
    'log_Income': np.log1p(county_2018['Median_Household_Income'])
})


# Scale data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_features)

best_dict = get_scores_and_labels(combinations, X_scaled)

print("Best Epsilon:", best_dict['best_epsilon'])
print("Best Min Samples:", best_dict['best_min_samples'])
print("Best Silhouette Score:", best_dict['best_score'])

Best Epsilon: 0.22894736842105262
Best Min Samples: 14
Best Silhouette Score: 0.6438001413570723


In [20]:
dbscan_model = DBSCAN(eps=best_dict['best_epsilon'], min_samples=best_dict['best_min_samples']).fit(X_scaled)
dbscan_model

,eps,np.float64(0....4736842105262)
,min_samples,np.int64(14)
,metric,'euclidean'
,metric_params,None
,algorithm,'auto'
,leaf_size,30
,p,None
,n_jobs,None


In [21]:
county_2018['cluster'] = dbscan_model.labels_
county_2018['cluster'].value_counts()

cluster
 0    2972
-1     229
 1      18
Name: count, dtype: int64

In [22]:
unemployment_rate, median_household_income = county_2018['Unemployment_rate'], county_2018['Median_Household_Income']

In [23]:
fig = px.scatter(x=unemployment_rate, y=median_household_income, color=county_2018['cluster'].astype(str), opacity=0.4,
                 title="DBSCAN Clustering - U.S. County Unemployment Rate vs. Median Household Income (2018)",
                 labels= {'x': 'Unemployment Rate (%)', 'y': 'Median Household Income (USD)'})
# fig.update_traces(marker=dict(size=3))
fig.show()

In [24]:
# Check cluster counts and noise percentage
print("Cluster Counts:")
print(county_2018['cluster'].value_counts())

print("\nNoise Percentage:")
print(f"{(county_2018['cluster'] == -1).mean() * 100:.2f}%")

Cluster Counts:
cluster
 0    2972
-1     229
 1      18
Name: count, dtype: int64

Noise Percentage:
7.11%


In [25]:
# Group by cluster to see the socio-economic profile of each group
cluster_summary = county_2018.groupby('cluster').agg(
    County_Count=('FIPS_Code', 'count'),
    Avg_Unemployment_Rate=('Unemployment_rate', 'mean'),
    Avg_Median_Income=('Median_Household_Income', 'mean'),
    Min_Income=('Median_Household_Income', 'min'),
    Max_Income=('Median_Household_Income', 'max')
).reset_index()

cluster_summary

,cluster,County_Count,Avg_Unemployment_Rate,Avg_Median_Income,Min_Income,Max_Income
0,-1,229,8.223722,58642.896651,28972.000000,167605.000000
1,0,2972,3.942136,63076.428496,36106.000000,127677.000000
2,1,18,9.922405,35752.022296,35487.754305,36437.890236


With a more refined epsilon search, the improved DBSCAN model had a higher silhouette score of ~0.65 compared to the original ~0.64, a reduction of categorized noise from 7.74% to 7.11%, and more data categorized for the economic hardship cluster (cluster 1).

##### 2019 Data

In [26]:
epsilons = np.linspace(0.15, 0.3, 20)
epsilons

array([0.15      , 0.15789474, 0.16578947, 0.17368421, 0.18157895,
       0.18947368, 0.19736842, 0.20526316, 0.21315789, 0.22105263,
       0.22894737, 0.23684211, 0.24473684, 0.25263158, 0.26052632,
       0.26842105, 0.27631579, 0.28421053, 0.29210526, 0.3       ])

In [27]:
min_samples = np.arange(10, 30, 4)
min_samples

array([10, 14, 18, 22, 26])

In [28]:
combinations = list(itertools.product(epsilons, min_samples))
N = len(combinations)
N

100

In [29]:
X_features = pd.DataFrame({
    'Unemploymnt_rate': county_2019['Unemployment_rate'],
    'log_Income': np.log1p(county_2019['Median_Household_Income'])
})


# Scale data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_features)

best_dict = get_scores_and_labels(combinations, X_scaled)

print("Best Epsilon:", best_dict['best_epsilon'])
print("Best Min Samples:", best_dict['best_min_samples'])
print("Best Silhouette Score:", best_dict['best_score'])

Best Epsilon: 0.3
Best Min Samples: 14
Best Silhouette Score: 0.650957601356923


In [30]:
best_dict

{'best_epsilon': np.float64(0.3),
 'best_min_samples': np.int64(14),
 'best_labels': array([ 0,  0,  0, ..., -1, -1, -1], shape=(3219,)),
 'best_score': 0.650957601356923}

In [31]:
dbscan_model = DBSCAN(eps=best_dict['best_epsilon'], min_samples=best_dict['best_min_samples']).fit(X_scaled)
dbscan_model

,eps,np.float64(0.3)
,min_samples,np.int64(14)
,metric,'euclidean'
,metric_params,None
,algorithm,'auto'
,leaf_size,30
,p,None
,n_jobs,None


In [32]:
county_2019['cluster'] = dbscan_model.labels_
county_2019['cluster'].value_counts()

cluster
 0    3052
-1     150
 1      17
Name: count, dtype: int64

In [33]:
unemployment_rate, median_household_income = county_2019['Unemployment_rate'], county_2019['Median_Household_Income']

In [34]:
fig = px.scatter(x=unemployment_rate, y=median_household_income, color=county_2018['cluster'].astype(str), opacity=0.4,
                 title="DBSCAN Clustering - U.S. County Unemployment Rate vs. Median Household Income (2019)",
                 labels= {'x': 'Unemployment Rate (%)', 'y': 'Median Household Income (USD)'})
# fig.update_traces(marker=dict(size=3))
fig.show()

In [35]:
# Check cluster counts and noise percentage
print("Cluster Counts:")
print(county_2019['cluster'].value_counts())

print("\nNoise Percentage:")
print(f"{(county_2019['cluster'] == -1).mean() * 100:.2f}%")

Cluster Counts:
cluster
 0    3052
-1     150
 1      17
Name: count, dtype: int64

Noise Percentage:
4.66%


In [36]:
# Group by cluster to see the socio-economic profile of each group
cluster_summary = county_2019.groupby('cluster').agg(
    County_Count=('FIPS_Code', 'count'),
    Avg_Unemployment_Rate=('Unemployment_rate', 'mean'),
    Avg_Median_Income=('Median_Household_Income', 'mean'),
    Min_Income=('Median_Household_Income', 'min'),
    Max_Income=('Median_Household_Income', 'max')
).reset_index()

cluster_summary

,cluster,County_Count,Avg_Unemployment_Rate,Avg_Median_Income,Min_Income,Max_Income
0,-1,150,8.291549,63832.348133,28972.0,167605.000000
1,0,3052,3.862051,62697.707563,33290.0,131562.000000
2,1,17,10.400309,35917.408720,33148.0,39000.675319


##### 2020 Data

In [37]:
epsilons = np.linspace(0.15, 0.3, 20)
epsilons

array([0.15      , 0.15789474, 0.16578947, 0.17368421, 0.18157895,
       0.18947368, 0.19736842, 0.20526316, 0.21315789, 0.22105263,
       0.22894737, 0.23684211, 0.24473684, 0.25263158, 0.26052632,
       0.26842105, 0.27631579, 0.28421053, 0.29210526, 0.3       ])

In [38]:
min_samples = np.arange(10, 30, 4)
min_samples

array([10, 14, 18, 22, 26])

In [39]:
combinations = list(itertools.product(epsilons, min_samples))
N = len(combinations)
N

100

In [40]:
X_features = pd.DataFrame({
    'Unemploymnt_rate': county_2020['Unemployment_rate'],
    'log_Income': np.log1p(county_2020['Median_Household_Income'])
})


# Scale data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_features)

best_dict = get_scores_and_labels(combinations, X_scaled)

print("Best Epsilon:", best_dict['best_epsilon'])
print("Best Min Samples:", best_dict['best_min_samples'])
print("Best Silhouette Score:", best_dict['best_score'])

Best Epsilon: 0.28421052631578947
Best Min Samples: 14
Best Silhouette Score: 0.4238569914776957


In [41]:
best_dict

{'best_epsilon': np.float64(0.28421052631578947),
 'best_min_samples': np.int64(14),
 'best_labels': array([0, 0, 0, ..., 0, 0, 0], shape=(3142,)),
 'best_score': 0.4238569914776957}

In [42]:
dbscan_model = DBSCAN(eps=best_dict['best_epsilon'], min_samples=best_dict['best_min_samples']).fit(X_scaled)
dbscan_model

,eps,np.float64(0....1052631578947)
,min_samples,np.int64(14)
,metric,'euclidean'
,metric_params,None
,algorithm,'auto'
,leaf_size,30
,p,None
,n_jobs,None


In [43]:
county_2020['cluster'] = dbscan_model.labels_
county_2020['cluster'].value_counts()

cluster
 0    2941
-1     191
 1      10
Name: count, dtype: int64

In [44]:
unemployment_rate, median_household_income = county_2020['Unemployment_rate'], county_2020['Median_Household_Income']

In [45]:
fig = px.scatter(x=unemployment_rate, y=median_household_income, color=county_2020['cluster'].astype(str), opacity=0.4,
                 title="DBSCAN Clustering - U.S. County Unemployment Rate vs. Median Household Income (2020)",
                 labels= {'x': 'Unemployment Rate (%)', 'y': 'Median Household Income (USD)'})
# fig.update_traces(marker=dict(size=3))
fig.show()

In [46]:
# Check cluster counts and noise percentage
print("Cluster Counts:")
print(county_2020['cluster'].value_counts())

print("\nNoise Percentage:")
print(f"{(county_2020['cluster'] == -1).mean() * 100:.2f}%")

Cluster Counts:
cluster
 0    2941
-1     191
 1      10
Name: count, dtype: int64

Noise Percentage:
6.08%


In [47]:
# Group by cluster to see the socio-economic profile of each group
cluster_summary = county_2020.groupby('cluster').agg(
    County_Count=('FIPS_Code', 'count'),
    Avg_Unemployment_Rate=('Unemployment_rate', 'mean'),
    Avg_Median_Income=('Median_Household_Income', 'mean'),
    Min_Income=('Median_Household_Income', 'min'),
    Max_Income=('Median_Household_Income', 'max')
).reset_index()

cluster_summary

,cluster,County_Count,Avg_Unemployment_Rate,Avg_Median_Income,Min_Income,Max_Income
0,-1,191,10.365683,73789.272251,28972.0,167605.0
1,0,2941,6.535870,62405.539272,35152.0,112218.0
2,1,10,4.944191,116341.500000,112154.0,120301.0


### Hierarchical Agglomerative Clustering (HAC)

##### 2018 Data

In [48]:
X_features = pd.DataFrame({
    'Unemploymnt_rate': county_2018['Unemployment_rate'],
    'log_Income': np.log1p(county_2018['Median_Household_Income'])
})

In [49]:
# Find optimal K

results = []
for k in range(2, 9):
    hac = AgglomerativeClustering(n_clusters=k, linkage='ward')
    labels = hac.fit_predict(X_features)
    score = silhouette_score(X_features, labels)
    results.append({'K': k, 'score': score})

hac_scores = pd.DataFrame(results)
hac_scores

,K,score
0,2,0.719882
1,3,0.517063
2,4,0.524559
3,5,0.434716
4,6,0.420058
5,7,0.421523
6,8,0.421909


In [50]:
best_index = np.argmax(hac_scores['score'])
optimal_k = hac_scores.loc[best_index, 'K']
print(f"Optimal K: {optimal_k}")

Optimal K: 2


In [51]:
best_hac = AgglomerativeClustering(n_clusters=optimal_k, linkage='ward')
county_2018['hac_cluster'] = best_hac.fit_predict(X_features)

In [52]:
fig = px.scatter(county_2018, x='Unemployment_rate', y='Median_Household_Income', color=county_2018['hac_cluster'].astype(str), opacity=0.6,
                title=f"HAC Clustering - U.S. County Unemployment Rate vs. Median Household Income (2018)",
                labels={
                    'Unemployment_rate': 'Unemployment Rate (%)',
                    'Median_Household_Income': 'Median Household Income (USD)',
                    'color': 'HAC Tier'
                }
)
fig.show()

In [53]:
# Group by cluster to see the socio-economic profile of each group
cluster_summary = county_2018.groupby('hac_cluster').agg(
    County_Count=('FIPS_Code', 'count'),
    Avg_Unemployment_Rate=('Unemployment_rate', 'mean'),
    Avg_Median_Income=('Median_Household_Income', 'mean'),
    Min_Income=('Median_Household_Income', 'min'),
    Max_Income=('Median_Household_Income', 'max')
).reset_index()

cluster_summary

,hac_cluster,County_Count,Avg_Unemployment_Rate,Avg_Median_Income,Min_Income,Max_Income
0,0,2997,3.904298,63843.443147,30659.0,167605.0
1,1,222,9.354426,45932.910421,28972.0,90897.0


This HAC model is a very interpretable clustering result because it divides the counties into two categories: lower average unemployment and higher average unemployment. Cluster 0, the blue data, has an average unemployment rate of 3.90% with an average median income of $63,843. Cluster 1, the red data, has an average unemployment rate of 9.35% with an average median income of $45,932. This is a more interpretable result compared to the DBSCAN because both groups define two unique economic conditions: counties with lower unemployment and higher income; counties with higher unemployment and lower income.

##### 2019 Data

In [54]:
X_features = pd.DataFrame({
    'Unemploymnt_rate': county_2019['Unemployment_rate'],
    'log_Income': np.log1p(county_2019['Median_Household_Income'])
})

In [55]:
# Find optimal K

results = []
for k in range(2, 9):
    hac = AgglomerativeClustering(n_clusters=k, linkage='ward')
    labels = hac.fit_predict(X_features)
    score = silhouette_score(X_features, labels)
    results.append({'K': k, 'score': score})

hac_scores = pd.DataFrame(results)
hac_scores

,K,score
0,2,0.611323
1,3,0.587918
2,4,0.469547
3,5,0.481074
4,6,0.482211
5,7,0.404931
6,8,0.373523


In [56]:
best_index = np.argmax(hac_scores['score'])
optimal_k = hac_scores.loc[best_index, 'K']
print(f"Optimal K: {optimal_k}")

Optimal K: 2


In [57]:
best_hac = AgglomerativeClustering(n_clusters=optimal_k, linkage='ward')
county_2019['hac_cluster'] = best_hac.fit_predict(X_features)

In [58]:
fig = px.scatter(county_2019, x='Unemployment_rate', y='Median_Household_Income', color=county_2019['hac_cluster'].astype(str), opacity=0.6,
                title=f"HAC Clustering - U.S. County Unemployment Rate vs. Median Household Income (2019)",
                labels={
                    'Unemployment_rate': 'Unemployment Rate (%)',
                    'Median_Household_Income': 'Median Household Income (USD)',
                    'color': 'HAC Tier'
                }
)
fig.show()

In [59]:
# Group by cluster to see the socio-economic profile of each group
cluster_summary = county_2019.groupby('hac_cluster').agg(
    County_Count=('FIPS_Code', 'count'),
    Avg_Unemployment_Rate=('Unemployment_rate', 'mean'),
    Avg_Median_Income=('Median_Household_Income', 'mean'),
    Min_Income=('Median_Household_Income', 'min'),
    Max_Income=('Median_Household_Income', 'max')
).reset_index()

cluster_summary

,hac_cluster,County_Count,Avg_Unemployment_Rate,Avg_Median_Income,Min_Income,Max_Income
0,0,692,6.616585,51105.197523,28972.0,106560.0
1,1,2527,3.414658,65759.420247,33090.0,167605.0


##### 2020 Data

In [60]:
X_features = pd.DataFrame({
    'Unemploymnt_rate': county_2020['Unemployment_rate'],
    'log_Income': np.log1p(county_2020['Median_Household_Income'])
})

In [61]:
# Find optimal K

results = []
for k in range(2, 9):
    hac = AgglomerativeClustering(n_clusters=k, linkage='ward')
    labels = hac.fit_predict(X_features)
    score = silhouette_score(X_features, labels)
    results.append({'K': k, 'score': score})

hac_scores = pd.DataFrame(results)
hac_scores

,K,score
0,2,0.492523
1,3,0.478828
2,4,0.468801
3,5,0.455569
4,6,0.458776
5,7,0.416889
6,8,0.411523


In [62]:
best_index = np.argmax(hac_scores['score'])
optimal_k = hac_scores.loc[best_index, 'K']
print(f"Optimal K: {optimal_k}")

Optimal K: 2


In [63]:
best_hac = AgglomerativeClustering(n_clusters=optimal_k, linkage='ward')
county_2020['hac_cluster'] = best_hac.fit_predict(X_features)

In [64]:
fig = px.scatter(county_2020, x='Unemployment_rate', y='Median_Household_Income', color=county_2020['hac_cluster'].astype(str), opacity=0.6,
                title=f"HAC Clustering - U.S. County Unemployment Rate vs. Median Household Income (2020)",
                labels={
                    'Unemployment_rate': 'Unemployment Rate (%)',
                    'Median_Household_Income': 'Median Household Income (USD)',
                    'color': 'HAC Tier'
                }
)
fig.show()

In [65]:
# Group by cluster to see the socio-economic profile of each group
cluster_summary = county_2020.groupby('hac_cluster').agg(
    County_Count=('FIPS_Code', 'count'),
    Avg_Unemployment_Rate=('Unemployment_rate', 'mean'),
    Avg_Median_Income=('Median_Household_Income', 'mean'),
    Min_Income=('Median_Household_Income', 'min'),
    Max_Income=('Median_Household_Income', 'max')
).reset_index()

cluster_summary

,hac_cluster,County_Count,Avg_Unemployment_Rate,Avg_Median_Income,Min_Income,Max_Income
0,0,1811,8.236182,61217.102706,28972.0,150502.0
1,1,1331,4.759996,66061.370398,35798.0,167605.0
